In [8]:
import geopandas as gpd
import os

operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "walk"  # walk or bike
type="geo_weighted" # rural_loisir or urbain_fonctionnel
index_col = f"{network}_index" 

input_file_path = '../../Data/input'
output_step1_path=f"../../Data/output/{network}/{territory}/step-1"
output_step2_path=f"../../Data/output/{network}/{territory}/step-2"
output_step3_path=f"../../Data/output/{network}/{territory}/step-3_{type}"
save_path = f"../../Data/output/{network}/{territory}/step-3_{type}"

#Read the files
index = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')
index = index.to_crs(operation_crs)

zones_girec = gpd.read_file(f'{input_file_path}/network_agreg/GEO_GIREC-SHP/GEO_GIREC.shp')
zones_girec = zones_girec.to_crs(operation_crs)

gg_perimeter = gpd.read_file(f'{input_file_path}/network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP/AGGLO_PERIMETRE_AVEC_LAC.shp')
gg_perimeter = gg_perimeter.to_crs(operation_crs)

iris = gpd.read_file(f'{input_file_path}/network_agreg/CONTOURS-IRIS-PE_3-0__GPKG_LAMB93_FXX_2025-01-01/CONTOURS-IRIS-PE/1_DONNEES_LIVRAISON_2025-09-00130/CONTOURS-IRIS-PE_3-0_GPKG_LAMB93_FXX-ED2025-01-01/contours-iris-pe.gpkg')
iris = iris.to_crs(operation_crs)

communes_contours = gpd.read_file(f'{input_file_path}/network_agreg/swissboundaries3d_2026-01_2056_5728/swissBOUNDARIES3D_1_5_TLM_HOHEITSGEBIET.shp')
communes_contours = communes_contours.to_crs(operation_crs)

cantons_contours = gpd.read_file(f'{input_file_path}/network_agreg/swissboundaries3d_2026-01_2056_5728/swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.shp')
cantons_contours = cantons_contours.to_crs(operation_crs)

agglo_carreau = gpd.read_file(f'{input_file_path}/network_agreg/AGGLO_CARREAU_200-SHP/AGGLO_CARREAU_200.shp')
agglo_carreau = agglo_carreau.to_crs(operation_crs)

lac_contours = gpd.read_file(f'{input_file_path}/network_agreg/GEO_LAC_LEMAN-SHP/GEO_LAC_LEMAN.shp')
lac_contours = lac_contours.to_crs(operation_crs)

import pandas as pd
# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)

Set parameters : GG or GE, bike or walk


**Preparer la couche Secteur Infracommunal GG**

In [9]:
gg_geometry = gg_perimeter.union_all()
leman_contour = lac_contours[lac_contours['NOM'] == 'Léman'].copy()
gg_land_perimeter = gpd.GeoDataFrame(geometry=[gg_geometry.difference(leman_contour.union_all())], crs=operation_crs)
gg_land_geometry = gg_land_perimeter.union_all()
vaud_canton = cantons_contours[cantons_contours['NAME'] == 'Vaud'].copy()

geneva_contours = gpd.clip(zones_girec, gg_land_perimeter)[['OBJECTID', 'NOM', 'geometry']].copy()
geneva_contours['source'] = 'zones_girec'
geneva_contours['country'] = 'CH'
geneva_contours['territory'] = 'GE'
geneva_contours['unit_id'] = geneva_contours['OBJECTID'].astype(str)
geneva_contours['unit_name'] = geneva_contours['NOM']
geneva_contours['commune_name'] = geneva_contours['NOM']

iris_gg = gpd.clip(iris, gg_land_perimeter)[['code_iris', 'nom_iris', 'nom_commune', 'geometry']].copy()
iris_gg['source'] = 'iris'
iris_gg['country'] = 'FR'
iris_gg['territory'] = 'FR'
iris_gg['unit_id'] = iris_gg['code_iris'].astype(str)
iris_gg['unit_name'] = iris_gg['nom_iris']
iris_gg['commune_name'] = iris_gg['nom_commune']

vaud_communes = communes_contours[
    (communes_contours['KANTONSNUM'] == 22)
    & communes_contours.intersects(gg_land_geometry)
    & communes_contours.intersects(vaud_canton.union_all())
].copy()
vaud_communes = gpd.clip(vaud_communes, gg_land_perimeter)[['BFS_NUMMER', 'NAME', 'geometry']].copy()
vaud_communes['source'] = 'communes'
vaud_communes['country'] = 'CH'
vaud_communes['territory'] = 'VD'
vaud_communes['unit_id'] = vaud_communes['BFS_NUMMER'].astype(int).astype(str)
vaud_communes['unit_name'] = vaud_communes['NAME']
vaud_communes['commune_name'] = vaud_communes['NAME']

gg_contours = gpd.GeoDataFrame(
    pd.concat([
        geneva_contours[['source', 'country', 'territory', 'unit_id', 'unit_name', 'commune_name', 'geometry']],
        iris_gg[['source', 'country', 'territory', 'unit_id', 'unit_name', 'commune_name', 'geometry']],
        vaud_communes[['source', 'country', 'territory', 'unit_id', 'unit_name', 'commune_name', 'geometry']],
    ], ignore_index=True),
    crs=operation_crs,
)

gg_infra_communal = gg_contours.copy()

print(gg_contours[['source', 'territory']].value_counts())
print(f'Total units: {len(gg_contours)}')
gg_contours.head()


source       territory
zones_girec  GE           475
iris         FR           223
communes     VD            54
Name: count, dtype: int64
Total units: 752


,source,country,territory,unit_id,unit_name,commune_name,geometry
0,zones_girec,CH,GE,56,Chancy - Les Bois,Chancy - Les Bois,"POLYGON ((2487352.337 1111085.089, 2487382.897..."
1,zones_girec,CH,GE,174,Les Lolliets,Les Lolliets,"POLYGON ((2492383.158 1111518.688, 2492434.096..."
2,zones_girec,CH,GE,172,Entre-Deux-Vignobles,Entre-Deux-Vignobles,"POLYGON ((2492506.369 1111964.599, 2492525.998..."
3,zones_girec,CH,GE,211,Le Renfort,Le Renfort,"POLYGON ((2489299.358 1112090.942, 2489304.058..."
4,zones_girec,CH,GE,35,Les Creux-du-Loup,Les Creux-du-Loup,"POLYGON ((2489141.405 1112132.614, 2489172.757..."


### ⚠️ CORRECTION (Claude, 2026-09-23) : méthode d'agrégation

L'agrégation d'origine est remplacée par une version corrigée **dans les trois cellules qui suivent** (GIREC, GG infra-communal, Carreau 200). Chaque changement est encadré par `# >>> CORRECTION` / `# <<< FIN CORRECTION`, avec l'ancienne ligne gardée en commentaire (`# AVANT :`).

**Problème 1 : `predicate="within"` supprimait des tronçons.**
Un tronçon à cheval sur deux polygones n'est « within » d'aucun des deux, donc il sortait complètement du calcul. Avec des tronçons ≤ 50 m et des carreaux de 200 m, ça représente environ un quart du réseau (estimation 4L/(πa)).
→ Désormais, chaque tronçon est **découpé** par les polygones qu'il intersecte, et chaque morceau est pondéré par la **longueur réellement dans le polygone** (`length_in_zone`) au lieu de `length`.

**Problème 2 : les NaN (masque RD hors agglo) biaisaient la moyenne.**
`df[cols].multiply(w).sum()` ignore les NaN au numérateur, mais `df[w].sum()` gardait leur longueur au dénominateur. Conséquences :
- les polygones en partie masqués avaient un indice tiré vers le bas ;
- les polygones entièrement masqués avaient un indice = **0** au lieu de **NaN**, et passaient donc le `dropna`.

→ Désormais, `weighted_mean` ne met au dénominateur que la longueur des tronçons dont la valeur n'est pas NaN, colonne par colonne. Si une colonne n'a que des NaN dans un polygone, le résultat est NaN.

**Changement annexe :** `cols_to_agg` ne dépend plus de l'ordre des colonnes (`cols[3:]`). Il prend explicitement toutes les colonnes numériques sauf les identifiants, la longueur et la géométrie. Le contenu est le même qu'avant.

Rien d'autre n'est modifié : entrées, préparation des contours, jointures des résultats, `dropna`, export.

In [10]:
# >>> CORRECTION (Claude, 2026-09-23) : nouvelles fonctions d'agrégation, voir la note ci-dessus
import numpy as np

def split_segments_by_zones(segments, zones, zone_id_col):
    """Découpe chaque tronçon par les polygones qu'il intersecte.
    Remplace gpd.sjoin(..., predicate="within"), qui supprimait les tronçons à cheval sur une limite.
    Renvoie un morceau par couple (tronçon, polygone), avec la longueur du morceau dans 'length_in_zone'."""
    zones = zones[[zone_id_col, "geometry"]]
    assert zones.index.is_unique, "L'index des polygones doit être unique"
    pieces = gpd.sjoin(segments, zones, how="inner", predicate="intersects")
    zone_geom = gpd.GeoSeries(zones.geometry.loc[pieces["index_right"]].values,
                              index=pieces.index, crs=zones.crs)
    pieces = pieces.set_geometry(pieces.geometry.intersection(zone_geom))
    pieces["length_in_zone"] = pieces.geometry.length
    # On enlève les contacts ponctuels (tronçon qui touche seulement la limite : longueur 0)
    return pieces[pieces["length_in_zone"] > 0].drop(columns="index_right")

def weighted_mean(df, cols, weight_col):
    """Moyenne pondérée qui ignore les NaN au numérateur ET au dénominateur.
    AVANT : (df[cols] * w).sum() / w.sum() -> la longueur des NaN restait au dénominateur, et un groupe tout NaN donnait 0."""
    x = df[cols].astype(float)   # bool -> 0/1
    w = df[weight_col]
    num = x.multiply(w, axis=0).sum()
    den = x.notna().multiply(w, axis=0).sum()
    return num / den.replace(0, np.nan)
# <<< FIN CORRECTION

**GIREC**

In [11]:
# Spatial join
# >>> CORRECTION : découpage des tronçons au lieu de "within" (les tronçons à cheval sur une limite étaient perdus)
# AVANT : segments_girec = gpd.sjoin(index, zones_girec, how="inner", predicate="within")
segments_girec = split_segments_by_zones(index, zones_girec, "OBJECTID")
# <<< FIN CORRECTION

# Columns to aggregate
cols = index.columns.to_list()

# >>> CORRECTION : weighted_mean est maintenant définie dans la cellule "CORRECTION" au-dessus (version qui gère les NaN)
# AVANT :
# def weighted_mean(df, cols, weight_col):
#     return (df[cols].multiply(df[weight_col], axis=0).sum() / df[weight_col].sum())
# <<< FIN CORRECTION

# >>> CORRECTION : liste explicite, ne dépend plus de l'ordre des colonnes
# AVANT : cols_to_agg = cols[3:]  # tes colonnes d'indicateurs
non_indicator_cols = {"segment_id", "length", "u", "v", "geometry"}
# (bool inclus : rd_hors_agglo_masked -> part de longueur masquée ; urbain -> part de réseau urbain)
cols_to_agg = [c for c in index.select_dtypes(["number", "bool"]).columns if c not in non_indicator_cols]
# <<< FIN CORRECTION

# Calcul pondéré
girec_stats = (
    segments_girec
        # >>> CORRECTION : sélection des colonnes (plus de DeprecationWarning) + poids = longueur dans la zone
        # AVANT : .groupby("OBJECTID")
        # AVANT : .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
        .groupby("OBJECTID")[cols_to_agg + ["length_in_zone"]]
        .apply(lambda d: weighted_mean(d, cols_to_agg, "length_in_zone"))
        # <<< FIN CORRECTION
        .reset_index()
)

# Merge back with zones_mmt polygons
zones_girec = zones_girec.merge(girec_stats, on="OBJECTID", how="left")

#Drop nan values 
zones_girec = zones_girec.dropna(subset=[index_col])

**GG infra-communal**

In [12]:
# Spatial join: assign each segment to a gg infra communal unit
# >>> CORRECTION : découpage des tronçons au lieu de "within"
# AVANT : segments_gg_infra_communal = gpd.sjoin(index, gg_infra_communal, how="inner", predicate="within")
segments_gg_infra_communal = split_segments_by_zones(index, gg_infra_communal, "unit_id")
# <<< FIN CORRECTION

# Aggregate by weighted mean
gg_infra_communal_stats = (
    segments_gg_infra_communal
    # >>> CORRECTION : poids = longueur dans la zone, weighted_mean qui gère les NaN
    # AVANT : .groupby("unit_id")
    # AVANT : .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .groupby("unit_id")[cols_to_agg + ["length_in_zone"]]
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length_in_zone"))
    # <<< FIN CORRECTION
    .reset_index()
)

# Merge back to gg infra communal polygons
gg_infra_communal = gg_infra_communal.merge(gg_infra_communal_stats, on="unit_id", how="left")

# Drop rows with missing values
gg_infra_communal = gg_infra_communal.dropna(subset=[index_col])

**Carreau 200**

In [13]:
# Spatial join: assign each segment to a carreau (grid cell)
# >>> CORRECTION : découpage des tronçons au lieu de "within" (environ 25 % du réseau perdu avec des carreaux de 200 m)
# AVANT : segments_carreau = gpd.sjoin(index, agglo_carreau, how="inner", predicate="within")
segments_carreau = split_segments_by_zones(index, agglo_carreau, "GRID_ID")
# <<< FIN CORRECTION

# Aggregate by mean
carreau_stats = (
    segments_carreau
    # >>> CORRECTION : poids = longueur dans le carreau, weighted_mean qui gère les NaN
    # AVANT : .groupby("GRID_ID")
    # AVANT : .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .groupby("GRID_ID")[cols_to_agg + ["length_in_zone"]]
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length_in_zone"))
    # <<< FIN CORRECTION
    .reset_index()
)

# Merge back to grid polygons
agglo_carreau = agglo_carreau.merge(carreau_stats, on="GRID_ID", how="left")

# Drop rows with missing values (optional)
# NB (correction) : les carreaux entièrement masqués RD ont maintenant index_col = NaN et sont donc enlevés ici
agglo_carreau = agglo_carreau.dropna(subset=[index_col])

**Export**

In [14]:
#save the file 
zones_girec.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_girec.gpkg"), driver="GPKG")
zones_girec.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_girec.parquet')

gg_infra_communal.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_gg_infra_communal.gpkg"), driver="GPKG")
gg_infra_communal.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_gg_infra_communal.parquet')

agglo_carreau.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_carreau200.gpkg"), driver="GPKG")
agglo_carreau.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_carreau200.parquet')